<!-- applied-genai-header -->
<div style="background: linear-gradient(135deg, #1e4b8f 0%, #2d6cb8 100%); color: white; padding: 24px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; margin-bottom: 16px;">
  <div style="font-size: 12px; opacity: 0.85; letter-spacing: 1.5px; text-transform: uppercase;">Applied Generative AI · IE 5373</div>
  <h1 style="margin: 8px 0 4px 0; font-size: 28px; font-weight: 600;">M3 Lab 1 — Prompting Strategies</h1>
  <div style="font-size: 14px; opacity: 0.9;">Zero-shot, few-shot, chain-of-thought, role/task/format scaffolds</div>
  <div style="margin-top: 12px; font-size: 12px; opacity: 0.8;">Prof. Mohammad Dehghani · Northeastern University</div>
</div>

> **📌 Note on models.** This lab references specific LLM versions (e.g. `gpt-5`, `gpt-5-mini`).
> Models update quickly — you are welcome (and encouraged) to swap in any newer OpenAI / Anthropic / Google model you have access to.
> The default model is set in one place: `DEFAULT_CHAT_MODEL` inside `utils.py`. Change it there and every cell follows.


In [1]:
# === Shared lab setup: utils.py + API key + sticky lab pill ===
# Downloads the shared utilities (pretty_print, model constants, key loader,
# lab_pill) from the AppliedGenAI repo so every notebook stays small and
# consistent. The API key is read from a Colab secret named OPENAI_API_KEY
# (set it once under Colab → 🔑 → "Notebook access" — same name in every lab).
import os
if not os.path.exists("utils.py"):
    !wget -q https://raw.githubusercontent.com/mdehghani86/AppliedGenAI/main/utils.py -O utils.py

from utils import (
    pretty_print,
    DEFAULT_CHAT_MODEL,   # e.g. "gpt-5"  — main reasoning model
    DEFAULT_MINI_MODEL,   # e.g. "gpt-5-mini"  — cheaper / faster default
    DEFAULT_EMBED_MODEL,  # e.g. "text-embedding-3-small"
    get_openai_key,
    lab_pill,
)

lab_pill('M3 Lab 1 — Prompting Strategies')        # sticky banner so you always see which lab you're in
get_openai_key(verify=True)    # loads the key + pings OpenAI to confirm it works


<a href="https://colab.research.google.com/github/mdehghani86/AppliedGenAI/blob/main/M3_Lab1_Prompting_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- Intro Section -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 30px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 32px;">Introduction to Prompting Strategies</h1>
    <p style="font-size: 18px; margin: 0;">Instructor: <strong>Dr. Dehghani</strong></p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Why It Matters Section -->
<div style="background: #ffffff; padding: 25px; border-radius: 10px; border-left: 6px solid #0055d4; box-shadow: 0 4px 8px rgba(0,0,0,0.05);">
    <h2 style="margin-top: 0; color: #001a70;">Why Prompting Strategies Matter</h2>
    <p style="font-size: 16px; line-height: 1.6;">
        Imagine you’re working with a junior engineer. You say:  
        <em>“Optimize the system.”</em><br>
        They’ll probably ask: <em>“Which system? Optimize for cost, speed, or energy? Any constraints?”</em> 🧐
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        Now try this instead:  
        <em>“Analyze the HVAC system and minimize energy consumption while keeping temperatures between 22-24°C. Provide a cost breakdown.”</em>  
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        That’s not just a prompt—it’s a <strong>clear strategy</strong> with defined objectives and boundaries.
        And that’s exactly what AI models need to perform at their best.
    </p>
</div>

<!-- Tip Section -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 30px;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Pro Tip</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        AI models appreciate well-structured instructions just like engineers appreciate complete design specs.
        Be specific, set clear goals, and watch the results improve!
    </p>
</div>

<!-- Upcoming Topics -->
<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">What’s Ahead</h3>
    <ul style="list-style: none; padding: 0; font-size: 16px; line-height: 1.8;">
        <li>📚 Basic Prompting Types</li>
        <li>🧩 Advanced Strategies</li>
        <li>📊 Application-Specific Techniques</li>
    </ul>
    <p style="font-size: 16px; color: #333;">Let’s engineer some powerful AI conversations! 🛠️</p>
</div>


<!-- Section Header -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 30px;">📚 Basic Prompting Types</h1>
</div>

<!-- Spacer -->
<div style="height: 25px;"></div>

<!-- Zero-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">1️⃣ Zero-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide only the task without any examples.  
        <strong>Use When:</strong> The task is simple and well-known by the model.  
        <em>Example:</em> “Translate 'Hello' to French.”
    </p>
</div>


In [2]:
# ==========================
# 📌 Set Up LLM and OpenAI API
# ==========================
# Import required libraries
from google.colab import userdata
import openai
import os

# Load the OpenAI API key securely from Colab secrets
api_key = userdata.get('OPENAI_API_KEY')

# Check that the API key was found
if api_key is None:
    raise ValueError("❌ API Key not found. Please store your OpenAI API key using Colab secrets.")

# Set API key as environment variable for OpenAI
os.environ["OPENAI_API_KEY"] = api_key

# Initialize OpenAI client
client = openai.OpenAI(api_key=api_key)

print("✅ OpenAI API Key successfully loaded and environment is ready!")

# ==========================
# 📌 Set LLM Model to GPT-3.5
# ==========================
# Define which LLM model to use
model_name = DEFAULT_MINI_MODEL

print(f"✅ LLM model set to: {model_name}")


✅ OpenAI API Key successfully loaded and environment is ready!
✅ LLM model set to: gpt-4o-mini


In [3]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot Test: Hidden Formula Sequence
# ==========================

hard_sequence_prompt_zero = (
    "The sequence is: 3, 12, 27, 48, 75, ___. What’s next?"
)

response_zero_hard = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": hard_sequence_prompt_zero}],
    temperature=0
)

print("🔹 LLM Response (Zero-Shot - Hard Sequence):\n")
pretty_print(response_zero_hard.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 LLM Response (Zero-Shot - Hard Sequence):




<!-- One-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">2️⃣ One-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide one clear example along with the instruction.  
        <strong>Use When:</strong> You want to guide the model’s behavior with a single example.  
        <em>Example:</em> “Translate 'Hello' to French: Bonjour. Now translate 'Goodbye'.”
    </p>
</div>


In [4]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot vs One-Shot Comparison: Alternating Pattern Sequence (Correct One-Shot)
# ==========================

model_name = DEFAULT_MINI_MODEL

# Zero-Shot Prompt (No Example)
zero_shot_prompt = (
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# One-Shot Prompt (One Example + New Question)
one_shot_prompt = (
    "Example:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Now solve this one:\n"
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run One-Shot
response_one = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": one_shot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response:\n" + "-"*40)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n\n🔹 One-Shot Response:\n" + "-"*40)
pretty_print(response_one.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Zero-Shot Response:
----------------------------------------




🔹 One-Shot Response:
----------------------------------------



<!-- Few-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">3️⃣ Few-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide multiple examples to clearly demonstrate the pattern.  
        <strong>Use When:</strong> The task is complex or requires understanding a specific format.  
        <em>Example:</em>  
        - “Translate 'Hello' to French: Bonjour.”  
        - “Translate 'Goodbye' to French: Au revoir.”  
        - “Translate 'Thank you' to French: Merci.”  
        Now translate 'Good night'.
    </p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Closing Tip -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Quick Reminder</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        The more complex the task, the more examples you should provide. But remember, too many examples can make prompts bulky and inefficient.
    </p>
</div>



In [5]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Few-Shot Prompting Example: Ultra-Hard Pattern (3 Hidden Rules)
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Best for complex reasoning

# Few-Shot Prompt with 2 Examples
few_shot_prompt = (
    "Example 1:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Example 2:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, ___. What’s next?\n"
    "Answer: 11.\n\n"
    "Now try this one:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, 11, ___, 16, 36. What number should replace the blank?"
)

# Run Few-Shot Prompt
response_few = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0
)

# Display Result
print("🔹 Few-Shot Prompting (Two Examples Provided):")
print("-" * 40)
pretty_print(response_few.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Few-Shot Prompting (Two Examples Provided):
----------------------------------------


## 🧠 Advanced Prompting Techniques  

Moving beyond basic prompting methods like zero-shot and few-shot, advanced strategies help enhance the reasoning and adaptability of large language models (LLMs). These techniques guide the model's thought process to handle complex tasks more effectively.

---

### 🔗 Chain-of-Thought (CoT) Prompting  

Chain-of-Thought prompting encourages models to **explain their intermediate reasoning steps**, leading to more transparent and accurate conclusions. By structuring prompts to include logical steps, CoT improves the model’s ability to solve complex reasoning tasks.

**Why is CoT Important?**  
- ✔️ Improves performance on multi-step reasoning tasks.  
- ✔️ Helps produce logically structured and coherent responses.  
- ✔️ Breaks down complex problems into manageable steps.

📖 **Reference:** [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)

---

*Next, explore practical examples of Chain-of-Thought prompting.*


In [7]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Chain-of-Thought Demonstration: Make 110 with Five 5's
# ==========================

model_name = DEFAULT_CHAT_MODEL

# Zero-Shot Prompt (No Reasoning Encouraged)
zero_shot_prompt = (
    "Use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110."
)

# Chain-of-Thought Prompt (Encourages Step-by-Step Reasoning)
cot_prompt = (
    "Let's solve this step by step.\n"
    "We need to use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110.\n"
    "Step 1: Think about how we can combine the 5's to form larger numbers (e.g., 55).\n"
    "Step 2: Try to combine them logically to reach 110.\n"
    "Now, provide the final equation and the answer."
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run Chain-of-Thought
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response (No Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n🔹 Chain-of-Thought Response (Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_cot.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Zero-Shot Response (No Reasoning Encouraged):
--------------------------------------------------



🔹 Chain-of-Thought Response (Reasoning Encouraged):
--------------------------------------------------


# ✋ Hands-On Experiment: Observations

📌 **Instructions:**
- Run your experiments by changing the model type, temperature, and prompt style.
- Below is my summary of observations across the prompts in this notebook.

---

- **Models Used:**
  `gpt-4o-mini` (DEFAULT_MINI_MODEL) for the basic prompting cells, and `gpt-4o` (DEFAULT_CHAT_MODEL) for the advanced reasoning cells. I re-ran the same Chain-of-Thought problems on both models to compare.

- **Temperature Setting:**
  `0.0` for deterministic zero-shot / one-shot / few-shot / CoT runs, and `0.7` inside the Self-Consistency loop. Higher temperature is required for Self-Consistency because at T=0 every sample would be identical, defeating the purpose of voting across multiple reasoning paths.

- **Zero-Shot Result:**
  Inconsistent. On the hidden-formula sequence (`3, 12, 27, 48, 75, ___`, rule `n²·(n+2)` → 108) the model often locked onto first-differences and returned a wrong "next term." On the alternating sequence (`1, 4, 2, 9, 3, 16, 4, ___`) zero-shot did not recognize the two interleaved series. On the five-5's = 110 task, zero-shot returned an expression that did not actually evaluate to 110.

- **One-Shot / Few-Shot Result:**
  Adding examples reframed the problem. One worked example on the alternating sequence signaled "look for interleaved series" and the model returned 25 (next square). Few-shot with two examples solved the 3-rule sequence that a single-example prompt could not.

- **Chain-of-Thought Result:**
  Clear win on the five-5's puzzle. The step-by-step scaffold ("build 55 first, then combine") forced the model to verify arithmetic at each step instead of guessing an expression. CoT also handled the train catch-up problem (correct answer: 5 PM) reliably at T=0.

- **Self-Consistency Result (surprising):**
  On the farmer's-sheep trap problem in my Cell 18 experiment, Self-Consistency actually **failed**. With `gpt-4o-mini` at T=0.7 the 5 samples were `['9','9','9','8','8']` and the majority vote returned **9**, which is wrong (the correct final answer is 7). With `gpt-4o` at T=0.7 the samples were `['9','9','7','7','9']` and again majority-voted to **9**. This was a useful counter-example to the common framing of Self-Consistency: majority voting only works when the model is *usually* right. When the model is *usually* making the same systematic mistake (in this case, stopping after Stage 1 of a two-stage problem), voting just amplifies that mistake.

- **Tree-of-Thought Result:**
  ToT was the only strategy that solved the farmer's-sheep trap reliably. Across all 4 conditions (2 models × 2 temperatures) it returned **7 every time**. The "generate 3 distinct solutions, then evaluate them" structure forces an explicit comparison step, and that evaluator catches errors like "wait, I only answered the first part." This was the strongest result in the notebook for me.

- **Key Takeaways:**
  1. **Prompt structure beats model size for reasoning tasks.** `gpt-4o-mini` + ToT beat `gpt-4o` + CoT on the trap problem. The cheaper model with a smarter scaffold outperformed the expensive model with a vague one.
  2. **Temperature is a diversity dial, not a quality dial.** CoT needs low temperature (commit to one careful chain). Self-Consistency *requires* high temperature (diversity is the whole point). Using the wrong setting breaks the strategy.
  3. **Self-Consistency is not magic.** It stabilizes existing capability — it doesn't unlock new capability. If the model is systematically wrong, voting will be systematically wrong.
  4. **ToT was the highest-leverage strategy** for the multi-stage problem I built. The explicit evaluator step is what catches errors CoT and SC both miss.
  5. **Few-shot has hidden cost.** Each example consumes tokens and biases the model. On problems whose structure differs from the examples, few-shot can hurt.

---

✍️ *Two models tested (gpt-4o-mini, gpt-4o), two temperature regimes (0.0 deterministic and 0.7 sampling), six prompting strategies compared. Biggest insight: Self-Consistency can fail on systematic errors; Tree-of-Thought was the only method that caught the multi-stage trap.*


## 🔁 Self-Consistency Prompting

While Chain-of-Thought (CoT) improves reasoning by encouraging step-by-step thinking, it may still produce **inconsistent or incorrect** answers, especially in complex scenarios.  
**Self-Consistency Prompting** enhances CoT by asking the model to **generate multiple reasoning paths** and then select the most common or consistent final answer.

### Why is Self-Consistency Useful?

- ✅ Reduces random reasoning errors.
- ✅ Boosts reliability on ambiguous or multi-path problems.
- ✅ Often improves performance on mathematical, logical, and symbolic tasks.

📖 **Reference**: [Self-Consistency Improves Chain of Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171)

---

*Next, we’ll see how Self-Consistency works in action using a complex reasoning example.*


In [8]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Comparing Chain-of-Thought vs. Self-Consistency Prompting
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Using GPT-4 for better reasoning

# Define the problem prompt
problem_prompt = (
    "If a train travels at 60 miles per hour and leaves at 2 PM, and another train leaves "
    "the same station at 3 PM traveling at 90 miles per hour, when will the second train catch up to the first?"
)

# Chain-of-Thought Prompt (Standard)
cot_prompt = (
    "Let's solve this step by step.\n"
    + problem_prompt
)

# Self-Consistency Prompt: Ask the model to produce multiple reasoning paths
def run_self_consistency(prompt, num_attempts=5):
    answers = []
    for _ in range(num_attempts):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Add randomness to explore different reasoning paths
        )
        answer = response.choices[0].message.content.strip()
        answers.append(answer)
    return answers

# Run Chain-of-Thought (Single Attempt)
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)
cot_answer = response_cot.choices[0].message.content.strip()

# Run Self-Consistency (Multiple Attempts)
sc_answers = run_self_consistency(cot_prompt, num_attempts=5)

# Simple Majority Vote to Find Most Consistent Answer
from collections import Counter
most_common_answer = Counter(sc_answers).most_common(1)[0]

# Display Results
print("🔹 Chain-of-Thought Response (Single Attempt):\n" + "-" * 50)
print(cot_answer)

print("\n🔹 Self-Consistency Responses (Multiple Attempts):\n" + "-" * 50)
for idx, ans in enumerate(sc_answers, 1):
    print(f"Attempt {idx}: {ans}")

print("\n🔹 Final Self-Consistency Selected Answer:\n" + "-" * 50)
print(f"Most Common Answer: {most_common_answer[0]}\nAppeared {most_common_answer[1]} times.")


🔹 Chain-of-Thought Response (Single Attempt):
--------------------------------------------------
To solve this problem, we need to determine when the second train will catch up to the first train. Let's break it down step by step:

1. **Determine the head start of the first train:**
   - The first train leaves at 2 PM traveling at 60 miles per hour.
   - By the time the second train leaves at 3 PM, the first train has been traveling for 1 hour.
   - In that 1 hour, the first train travels \(60 \text{ miles/hour} \times 1 \text{ hour} = 60 \text{ miles}\).

2. **Set up the equation for the distance traveled:**
   - Let \(t\) be the time in hours after 3 PM when the second train catches up to the first train.
   - In \(t\) hours, the second train travels \(90t\) miles.
   - In the same \(t\) hours, the first train travels an additional \(60t\) miles (since it continues traveling at 60 miles per hour).

3. **Write the equation for when the second train catches up:**
   - The second train 

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📚 Exploring More Advanced Prompting Strategies</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>🧩 Tree-of-Thought (ToT) Prompting:</strong> Explores multiple reasoning paths like a decision tree, helping the model evaluate and compare various solutions before choosing the best one.</li>
        <li><strong>🤖 ReAct (Reasoning and Acting) Prompting:</strong> Combines reasoning steps with actions, including API calls or external tool usage. Ideal for interactive agents and dynamic decision-making tasks.</li>
        <li><strong>🔄 Reflexion Prompting:</strong> Encourages the model to critique its own responses and iteratively improve them, simulating self-correction and learning.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">✋ Hands-On Task: Compare Prompting Strategies</h2>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <p style="font-size: 16px;">
        📌 <strong>Task Instructions:</strong><br>
        - Experiment with <strong>Self-Consistency</strong>, <strong>Tree-of-Thought</strong>, and <strong>ReAct</strong> prompting methods.<br>
        - Try to solve the following problem using each method and compare the results.
    </p>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <h3>🧠 <strong>Challenge Problem:</strong></h3>
    <p style="font-size: 16px;">A farmer has chickens and rabbits in a cage. There are 35 heads and 94 legs. How many chickens and rabbits are there?</p>
</div>

<div style="margin-top: 40px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Try different models (e.g., <code>gpt-3.5-turbo</code>, <code>gpt-4-turbo</code>, <code>gpt-o3</code>).</li>
        <li>Experiment with different temperatures (e.g., <code>0.0</code>, <code>0.5</code>, <code>0.7</code>).</li>
        <li>Use both direct prompts and advanced strategies like CoT, Self-Consistency, or ReAct.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">📖 Observations</h2>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>Model and Strategy Used:</strong><br>gpt-4o-mini and gpt-4o, each tested with Chain-of-Thought (T=0), Self-Consistency (T=0.7, 5 samples + majority vote), and Tree-of-Thought (3 candidate solutions + evaluator step).</li>
        <li><strong>Was the Correct Answer Found?</strong><br>Yes. The system 2C + 4R = 94, C + R = 35 yields <strong>23 chickens and 12 rabbits</strong>. CoT at T=0 found this on both models. Self-Consistency at T=0.7 also voted correctly. ToT was overkill for this problem — all 3 candidate solutions converged on the same answer, so the evaluator step added no value.</li>
        <li><strong>Key Takeaways:</strong><br>For well-posed algebra problems with a unique solution, plain CoT is sufficient and is the cheapest option. Self-Consistency adds no measurable accuracy here because the single-shot answer is already stable. ToT only earns its 3× API cost on problems with traps or multiple plausible interpretations — the sheep problem in Cell 18 is where ToT shines, not this one. The lesson: match the strategy to the failure mode, not to the problem's surface difficulty.</li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center;">
    ✍️ <em>Hint: Try breaking down the problem into equations or ask the model to explain its steps before giving the final answer. Notice which strategies lead to faster and more accurate results!</em>
</div>


In [10]:
# ==========================
# ✋ Chickens & Rabbits — comparing CoT, Self-Consistency, Tree-of-Thought
# ==========================
# Expected answer: 23 chickens and 12 rabbits.
# (2C + 4R = 94, C + R = 35 → C=23, R=12)

from openai import OpenAI
from collections import Counter
import re

client = OpenAI()

problem_cr = (
    "A farmer has chickens and rabbits in a cage. "
    "There are 35 heads and 94 legs in total. "
    "How many chickens and how many rabbits are there? "
    "On the final line, reply ONLY in the format: 'chickens=<int>, rabbits=<int>'."
)

def call_cr(model, prompt, temperature):
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return r.choices[0].message.content.strip()

def extract_pair(text):
    m = re.search(r"chickens\s*=\s*(\d+)\s*,\s*rabbits\s*=\s*(\d+)", text, re.IGNORECASE)
    return (int(m.group(1)), int(m.group(2))) if m else None

models = [DEFAULT_MINI_MODEL, DEFAULT_CHAT_MODEL]

print("=" * 70)
print(f"PROBLEM:\n{problem_cr}\n")
print("=" * 70)

results_cr = []
for model in models:
    print(f"\n--- Model: {model} ---")

    # 1) Chain-of-Thought (T=0)
    cot_prompt = "Let's solve this step by step.\n" + problem_cr
    cot_pair = extract_pair(call_cr(model, cot_prompt, 0.0))
    print(f"[CoT  T=0]              answer: {cot_pair}")

    # 2) Self-Consistency (T=0.7, 5 samples, majority vote)
    sc_samples = [extract_pair(call_cr(model, cot_prompt, 0.7)) for _ in range(5)]
    sc_clean = [s for s in sc_samples if s is not None]
    sc_majority = Counter(sc_clean).most_common(1)[0][0] if sc_clean else None
    print(f"[SC   T=0.7, n=5]       samples: {sc_clean}")
    print(f"                         majority: {sc_majority}")

    # 3) Tree-of-Thought (3 candidates + evaluator)
    tot_gen_prompt = (
        "Propose THREE distinct solution approaches to the following problem. "
        "Number them Solution 1, 2, 3. Each must show the algebra and end with "
        "the final answer in the format 'chickens=<int>, rabbits=<int>'.\n\n"
        f"Problem: {problem_cr}"
    )
    candidates = call_cr(model, tot_gen_prompt, 0.7)
    tot_eval_prompt = (
        "Below are three candidate solutions. Evaluate them, identify any logical "
        "errors, and select the one with the most internally consistent reasoning. "
        "Reply with ONLY the final line in the format 'chickens=<int>, rabbits=<int>'.\n\n"
        f"Problem: {problem_cr}\n\nCandidates:\n{candidates}"
    )
    tot_pair = extract_pair(call_cr(model, tot_eval_prompt, 0.0))
    print(f"[ToT (3 cand + eval)]   answer: {tot_pair}")

    results_cr.append({"model": model, "CoT": cot_pair, "SC": sc_majority, "ToT": tot_pair})

print("\n" + "=" * 70)
print("SUMMARY (expected: chickens=23, rabbits=12)")
print("=" * 70)
print(f"{'Model':<22} {'CoT':<22} {'SC':<22} {'ToT':<22}")
for r in results_cr:
    print(f"{r['model']:<22} {str(r['CoT']):<22} {str(r['SC']):<22} {str(r['ToT']):<22}")

PROBLEM:
A farmer has chickens and rabbits in a cage. There are 35 heads and 94 legs in total. How many chickens and how many rabbits are there? On the final line, reply ONLY in the format: 'chickens=<int>, rabbits=<int>'.


--- Model: gpt-4o-mini ---
[CoT  T=0]              answer: (23, 12)
[SC   T=0.7, n=5]       samples: [(23, 12), (23, 12), (23, 12), (23, 12), (23, 12)]
                         majority: (23, 12)
[ToT (3 cand + eval)]   answer: (23, 12)

--- Model: gpt-4o ---
[CoT  T=0]              answer: (23, 12)
[SC   T=0.7, n=5]       samples: [(23, 12), (23, 12), (23, 12), (23, 12), (23, 12)]
                         majority: (23, 12)
[ToT (3 cand + eval)]   answer: (23, 12)

SUMMARY (expected: chickens=23, rabbits=12)
Model                  CoT                    SC                     ToT                   
gpt-4o-mini            (23, 12)               (23, 12)               (23, 12)              
gpt-4o                 (23, 12)               (23, 12)               (23, 12

In [9]:
# ==========================
# ✋ Hands-On Code: Tree-of-Thought Experiment
# ==========================
# This cell extends the lab by implementing a simple Tree-of-Thought (ToT)
# prompting pattern and comparing it head-to-head against Chain-of-Thought
# and Self-Consistency on a single hard reasoning problem.
#
# Problem: a non-trivial logic puzzle where naive CoT often picks the wrong
# branch. ToT explicitly enumerates candidate solutions, evaluates each,
# and selects the best.
# ==========================

from openai import OpenAI
from collections import Counter
client = OpenAI()

# Models + temperatures we will sweep
models = [DEFAULT_MINI_MODEL, DEFAULT_CHAT_MODEL]
temperatures = [0.0, 0.7]

# A reasoning problem with a single correct numeric answer
problem = (
    "A farmer has 17 sheep. All but 9 run away. How many sheep does the farmer have left? "
    "Then: of those remaining, half are sold and 2 more are added. "
    "How many sheep does the farmer have at the end? "
    "Reply with ONLY a single integer on the final line."
)
# Correct answer: 9 - (9/2 rounded) + 2 ... but careful: the trick is "all BUT 9 run away" → 9 left.
# Then half of 9 are sold: 9 - 4 (or 5, depending on rounding) + 2.
# We'll let the model handle the rounding choice but score on the "all but 9 = 9 remain" trap.
# Expected final answer assuming integer rounding-down on "half sold": 9 - 4 + 2 = 7.
# (Models that miss the "all but 9" trap will start from 8 and get a different number.)

def call(model, prompt, temperature):
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return r.choices[0].message.content.strip()

def extract_final_int(text):
    """Grab the last integer in the response — the model is told to put it on the final line."""
    import re
    nums = re.findall(r"-?\d+", text)
    return nums[-1] if nums else None

# ---------- Strategy 1: Plain Chain-of-Thought ----------
cot_prompt = "Let's solve this step by step.\n" + problem

# ---------- Strategy 2: Self-Consistency over CoT (5 samples, majority vote) ----------
def self_consistency(prompt, model, n=5, temperature=0.7):
    answers = [extract_final_int(call(model, prompt, temperature)) for _ in range(n)]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None, []
    return Counter(answers).most_common(1)[0][0], answers

# ---------- Strategy 3: Tree-of-Thought ----------
# Step A: generate 3 candidate solutions independently
# Step B: ask the model to compare them and pick the best
tot_generate_prompt = (
    "Propose THREE distinct step-by-step solutions to the following problem. "
    "Number them Solution 1, Solution 2, Solution 3. For each, show the reasoning "
    "and the final integer answer.\n\nProblem: " + problem
)

tot_evaluate_prompt_template = (
    "Below are three candidate solutions to a reasoning problem. "
    "Evaluate them, identify any logical errors, and select the solution whose reasoning "
    "is most internally consistent. Reply with ONLY the final integer answer on the last line.\n\n"
    "Problem: {problem}\n\n"
    "Candidates:\n{candidates}"
)

def tree_of_thought(model, temperature=0.7):
    candidates_text = call(model, tot_generate_prompt, temperature)
    eval_prompt = tot_evaluate_prompt_template.format(problem=problem, candidates=candidates_text)
    final = call(model, eval_prompt, 0.0)  # deterministic evaluation step
    return final, candidates_text

# ---------- Run the sweep ----------
print("=" * 70)
print(f"PROBLEM:\n{problem}\n")
print("=" * 70)

results = []
for model in models:
    for temp in temperatures:
        print(f"\n--- Model: {model} | Temperature: {temp} ---")

        # CoT (single shot)
        cot_out = call(model, cot_prompt, temp)
        cot_ans = extract_final_int(cot_out)
        print(f"[CoT]              final answer: {cot_ans}")

        # Self-Consistency only meaningful at T>0
        if temp > 0:
            sc_ans, sc_all = self_consistency(cot_prompt, model, n=5, temperature=temp)
            print(f"[Self-Consistency] majority answer: {sc_ans}  (samples: {sc_all})")
        else:
            sc_ans = "n/a (T=0)"
            print(f"[Self-Consistency] skipped at T=0")

        # Tree-of-Thought
        tot_ans, _tot_cand = tree_of_thought(model, temperature=max(temp, 0.5))
        tot_int = extract_final_int(tot_ans)
        print(f"[Tree-of-Thought]  final answer: {tot_int}")

        results.append({
            "model": model, "temp": temp,
            "CoT": cot_ans, "SC": sc_ans, "ToT": tot_int,
        })

print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
print(f"{'Model':<22} {'Temp':<6} {'CoT':<8} {'SelfConsist':<14} {'ToT':<8}")
for r in results:
    print(f"{r['model']:<22} {str(r['temp']):<6} {str(r['CoT']):<8} {str(r['SC']):<14} {str(r['ToT']):<8}")

print("""
Interpretation:
- The "all but 9" wording is a classic trap. Weak models read "17 sheep, all but 9 run away" and
  compute 17 - 9 = 8 instead of recognizing that 9 are LEFT.
- CoT at T=0 should consistently produce one answer (right or wrong). At T=0.7 it becomes noisy.
- Self-Consistency stabilizes the noisy T=0.7 case by majority voting.
- Tree-of-Thought generates three independent attempts and then evaluates them — this catches
  the "all but 9" trap because at least one of the three branches usually parses the trick correctly.
""")


PROBLEM:
A farmer has 17 sheep. All but 9 run away. How many sheep does the farmer have left? Then: of those remaining, half are sold and 2 more are added. How many sheep does the farmer have at the end? Reply with ONLY a single integer on the final line.


--- Model: gpt-4o-mini | Temperature: 0.0 ---
[CoT]              final answer: 9
[Self-Consistency] skipped at T=0
[Tree-of-Thought]  final answer: 7

--- Model: gpt-4o-mini | Temperature: 0.7 ---
[CoT]              final answer: 8
[Self-Consistency] majority answer: 9  (samples: ['9', '9', '9', '8', '8'])
[Tree-of-Thought]  final answer: 7

--- Model: gpt-4o | Temperature: 0.0 ---
[CoT]              final answer: 9
[Self-Consistency] skipped at T=0
[Tree-of-Thought]  final answer: 7

--- Model: gpt-4o | Temperature: 0.7 ---
[CoT]              final answer: 7
[Self-Consistency] majority answer: 9  (samples: ['9', '9', '7', '7', '9'])
[Tree-of-Thought]  final answer: 7

SUMMARY TABLE
Model                  Temp   CoT      SelfConsist

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📌 Conclusion</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; line-height: 1.8;">
        In this hands-on exploration, different advanced prompting strategies were tested to solve reasoning-based challenges.
        Through experimenting with <strong>Chain-of-Thought (CoT)</strong>, <strong>Self-Consistency</strong>, and other methods,
        the following key insights were observed:
    </p>
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Advanced prompting techniques significantly improve model performance, especially on complex, multi-step problems.</li>
        <li>Changing the <strong>model type</strong> and <strong>temperature</strong> can drastically affect reasoning quality and creativity.</li>
        <li>Some strategies, like <strong>Self-Consistency</strong>, help reduce random errors by exploring multiple reasoning paths.</li>
        <li>For ambiguous or challenging problems, combining strategies (e.g., CoT + Self-Consistency) often leads to the most reliable results.</li>
    </ul>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; font-style: italic;">
        📖 <em>Remember: Prompt engineering is both an art and a science. The more you experiment, the better you understand how to guide LLMs effectively!</em>
    </p>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">✍️ Final Reflection</h3>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <p style="font-size: 16px;">
        The biggest thing I learned is that prompt structure is a multiplier on model capability, not a substitute for it — gpt-4o-mini with a well-built scaffold beat gpt-4o with a vague instruction on the same problem. Temperature is a diversity dial, not a quality dial: CoT wants T=0 (commit to one chain), Self-Consistency requires T≥0.5 (diversity is the point), and mismatching them breaks the strategy. My most surprising result was that Self-Consistency actually failed on the multi-stage sheep trap by majority-voting a systematic error — Tree-of-Thought was the only method that caught it, which convinced me that for problems with hidden traps, the explicit evaluator step in ToT is worth the extra API cost.
    </p>
</div>
